In [33]:
import os
import json
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
import torch.nn.functional as F

In [34]:
with open("data_split.json", "r") as f:
    split_data = json.load(f)

train_subjects = split_data["train"]
val_subjects = split_data["validation"]
test_subjects = split_data["test"]

print("Train:", len(train_subjects))
print("Validation:", len(val_subjects))
print("Test:", len(test_subjects))

Train: 1000
Validation: 125
Test: 126


In [35]:
train_data = r"Data/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData"

In [36]:
def preprocess_t2f(image):
    # Expected original BraTS shape
    if image.shape != (240, 240, 155):
        raise ValueError(
            f"Unexpected image shape: {image.shape}"
        )

    # Crop
    image = image[16:224, 8:232, :]

    # Pad depth from 155 to 160
    image = np.pad(
        image,
        ((0, 0), (0, 0), (2, 3)),
        mode="constant",
        constant_values=0
    )

    # Foreground mask before intensity transformation
    foreground = image > 0

    if not np.any(foreground):
        raise ValueError(
            "No foreground voxels found"
        )

    # Robust upper intensity limit
    upper = np.percentile(
        image[foreground],
        99.9
    )

    # Clip outliers
    image = np.clip(
        image,
        0,
        upper
    )

    # Scale to [0, 1]
    image = image / upper

    # Scale entire volume to [-1, 1]
    image = image * 2.0 - 1.0

    return image.astype(np.float32)

In [37]:
def preprocess_mask(mask):
    # Expected original BraTS shape
    if mask.shape != (240, 240, 155):
        raise ValueError(f"Unexpected mask shape: {mask.shape}")

    # Apply exactly the same spatial crop as T2f
    mask = mask[16:224, 8:232, :]

    # Apply exactly the same z-padding as T2f
    mask = np.pad(
        mask,
        ((0, 0), (0, 0), (2, 3)),
        mode="constant",
        constant_values=0
    )

    return mask.astype(np.int64)

In [38]:
def calculate_tumour_entropy(image, mask, num_bins=256):
    # Whole tumour = all non-zero tumour labels
    tumour_region = mask > 0

    if not np.any(tumour_region):
        raise ValueError("No tumour voxels found")

    tumour_values = image[tumour_region]

    # T2f has already been normalized to [0, 1]
    tumour_values = np.clip(tumour_values, 0.0, 1.0)

    hist, _ = np.histogram(
        tumour_values,
        bins=num_bins,
        range=(0.0, 1.0),
        density=False
    )

    probabilities = hist.astype(np.float64)
    probabilities = probabilities / probabilities.sum()

    probabilities = probabilities[probabilities > 0]

    entropy = -np.sum(
        probabilities * np.log2(probabilities)
    )

    return np.float32(entropy)

In [39]:
from torch.utils.data import Dataset, DataLoader

class BraTSDataset(Dataset):
    def __init__(self, subjects, data_dir):
        self.subjects = subjects
        self.data_dir = data_dir

    def __len__(self):
        return len(self.subjects)

    def __getitem__(self, idx):
        subject = self.subjects[idx]
        subject_path = os.path.join(self.data_dir, subject)

        files = os.listdir(subject_path)

        t2f_file = [f for f in files if "t2f" in f.lower()][0]
        seg_file = [f for f in files if "seg" in f.lower()][0]

        # Load T2f
        image = nib.load(
            os.path.join(subject_path, t2f_file)
        ).get_fdata()

        # Load segmentation
        mask = nib.load(
            os.path.join(subject_path, seg_file)
        ).get_fdata()

        # Apply preprocessing
        image = preprocess_t2f(image)
        mask = preprocess_mask(mask)

        # Calculate whole-tumour Shannon entropy
        entropy = calculate_tumour_entropy(
            image,
            mask
        )

        # Convert to tensors
        image = torch.from_numpy(
            image
        ).float().unsqueeze(0)

        mask = torch.from_numpy(
            mask
        ).long().unsqueeze(0)

        entropy = torch.tensor(
            entropy,
            dtype=torch.float32
        )

        return {
            "image": image,
            "mask": mask,
            "heterogeneity": entropy,
            "subject": subject
        }

In [40]:
train_dataset = BraTSDataset(
    subjects=train_subjects,
    data_dir=train_data
)

print("Dataset size:", len(train_dataset))

Dataset size: 1000


In [41]:
train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=0
)

In [42]:
sample = train_dataset[0]

print("Subject:", sample["subject"])
print("Image:", sample["image"].shape)
print("Mask:", sample["mask"].shape)
print("Mask labels:", torch.unique(sample["mask"]))
print("Heterogeneity:", sample["heterogeneity"])
print("Heterogeneity shape:", sample["heterogeneity"].shape)

Subject: BraTS-GLI-00240-000
Image: torch.Size([1, 208, 224, 160])
Mask: torch.Size([1, 208, 224, 160])
Mask labels: tensor([0, 1, 2, 3])
Heterogeneity: tensor(7.3900)
Heterogeneity shape: torch.Size([])


In [43]:
import numpy as np

entropy_values = []
tumour_volumes = []
subject_ids = []

for i in range(len(train_dataset)):
    sample = train_dataset[i]

    image_np = sample["image"][0].numpy()
    mask_np = sample["mask"][0].numpy()

    entropy = calculate_tumour_entropy(
        image_np,
        mask_np
    )

    # Whole tumour volume in voxels
    tumour_volume = np.sum(mask_np > 0)

    entropy_values.append(entropy)
    tumour_volumes.append(tumour_volume)
    subject_ids.append(sample["subject"])

entropy_values = np.array(entropy_values)
tumour_volumes = np.array(tumour_volumes)

print("Number of subjects:", len(entropy_values))

print("\nEntropy:")
print("Min:", entropy_values.min())
print("Max:", entropy_values.max())
print("Mean:", entropy_values.mean())
print("Median:", np.median(entropy_values))
print("Std:", entropy_values.std())

print("\nPercentiles:")
print("P10:", np.percentile(entropy_values, 10))
print("P25:", np.percentile(entropy_values, 25))
print("P50:", np.percentile(entropy_values, 50))
print("P75:", np.percentile(entropy_values, 75))
print("P90:", np.percentile(entropy_values, 90))

correlation = np.corrcoef(
    entropy_values,
    tumour_volumes
)[0, 1]

print("\nEntropy vs tumour volume correlation:")
print(correlation)

KeyboardInterrupt: 

In [44]:
timesteps = 1000

beta_start = 1e-4
beta_end = 0.02

betas = torch.linspace(beta_start, beta_end, timesteps)

alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)

sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)

In [45]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        device = t.device
        half_dim = self.dim // 2

        embeddings = math.log(10000) / (half_dim - 1)

        embeddings = torch.exp(
            torch.arange(half_dim, device=device) * -embeddings
        )

        embeddings = t[:, None] * embeddings[None, :]

        embeddings = torch.cat(
            (embeddings.sin(), embeddings.cos()),
            dim=1
        )

        return embeddings

In [46]:
class ResBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, time_dim):
        super().__init__()

        self.conv1 = nn.Conv3d(
            in_channels, out_channels,
            kernel_size=3, padding=1
        )

        self.conv2 = nn.Conv3d(
            out_channels, out_channels,
            kernel_size=3, padding=1
        )

        self.norm1 = nn.GroupNorm(
            num_groups=8,
            num_channels=out_channels
        )

        self.norm2 = nn.GroupNorm(
            num_groups=8,
            num_channels=out_channels
        )

        self.time_mlp = nn.Linear(
            time_dim,
            out_channels
        )

        if in_channels != out_channels:
            self.residual = nn.Conv3d(
                in_channels,
                out_channels,
                kernel_size=1
            )
        else:
            self.residual = nn.Identity()

    def forward(self, x, t):
        h = self.conv1(x)
        h = self.norm1(h)
        h = F.silu(h)

        time_emb = self.time_mlp(t)
        time_emb = time_emb[:, :, None, None, None]

        h = h + time_emb

        h = self.conv2(h)
        h = self.norm2(h)
        h = F.silu(h)

        return h + self.residual(x)

In [47]:
class DownBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, time_dim):
        super().__init__()

        self.resblock = ResBlock3D(
            in_channels,
            out_channels,
            time_dim
        )

        self.downsample = nn.Conv3d(
            out_channels,
            out_channels,
            kernel_size=4,
            stride=2,
            padding=1
        )

    def forward(self, x, t):
        h = self.resblock(x, t)

        down = self.downsample(h)

        return h, down


class UpBlock3D(nn.Module):
    def __init__(
        self,
        in_channels,
        skip_channels,
        out_channels,
        time_dim
    ):
        super().__init__()

        self.upsample = nn.ConvTranspose3d(
            in_channels,
            out_channels,
            kernel_size=4,
            stride=2,
            padding=1
        )

        self.resblock = ResBlock3D(
            out_channels + skip_channels,
            out_channels,
            time_dim
        )

    def forward(self, x, skip, t):
        x = self.upsample(x)

        x = torch.cat([x, skip], dim=1)

        x = self.resblock(x, t)

        return x

In [48]:
class ConditionalUNet3D(nn.Module):
    def __init__(
        self,
        image_channels=1,
        mask_channels=3,
        out_channels=1,
        base_channels=16,
        time_dim=128
    ):
        super().__init__()

        # Timestep embedding
        self.time_embedding = nn.Sequential(
            SinusoidalTimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim)
        )

        # Continuous heterogeneity scalar -> embedding
        self.heterogeneity_embedding = nn.Sequential(
            nn.Linear(1, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim)
        )

        # Noisy T2f + 3 tumour-mask channels
        total_in_channels = image_channels + mask_channels

        self.input_conv = nn.Conv3d(
            total_in_channels,
            base_channels,
            kernel_size=3,
            padding=1
        )

        self.down1 = DownBlock3D(
            base_channels,
            base_channels * 2,
            time_dim
        )

        self.down2 = DownBlock3D(
            base_channels * 2,
            base_channels * 4,
            time_dim
        )

        self.down3 = DownBlock3D(
            base_channels * 4,
            base_channels * 8,
            time_dim
        )

        self.mid = ResBlock3D(
            base_channels * 8,
            base_channels * 8,
            time_dim
        )

        self.up3 = UpBlock3D(
            in_channels=base_channels * 8,
            skip_channels=base_channels * 8,
            out_channels=base_channels * 4,
            time_dim=time_dim
        )

        self.up2 = UpBlock3D(
            in_channels=base_channels * 4,
            skip_channels=base_channels * 4,
            out_channels=base_channels * 2,
            time_dim=time_dim
        )

        self.up1 = UpBlock3D(
            in_channels=base_channels * 2,
            skip_channels=base_channels * 2,
            out_channels=base_channels,
            time_dim=time_dim
        )

        self.output_conv = nn.Conv3d(
            base_channels,
            out_channels,
            kernel_size=1
        )

    def forward(self, x, t, mask, heterogeneity):

        # -------------------------
        # 1. Multi-class mask 
        # -------------------------

        # Dataset gives:
        # mask shape = [B, 1, D, H, W]
        # labels = 0, 1, 2, 3

        mask = mask.squeeze(1)

        # One-hot gives [B, D, H, W, 4]
        mask_onehot = F.one_hot(
            mask.long(),
            num_classes=4
        )

        # -> [B, 4, D, H, W]
        mask_onehot = mask_onehot.permute(
            0, 4, 1, 2, 3
        ).float()

        # Remove background channel
        # -> [B, 3, D, H, W]
        mask_onehot = mask_onehot[:, 1:, ...]

        # Combine noisy T2f + tumour mask
        x = torch.cat(
            [x, mask_onehot],
            dim=1
        )

        # -------------------------
        # 2. Timestep embedding
        # -------------------------

        t_emb = self.time_embedding(t)

        # -------------------------
        # 3. Heterogeneity embedding
        # -------------------------

        # DataLoader gives approximately [B]
        heterogeneity = heterogeneity.float().view(-1, 1)

        h_emb = self.heterogeneity_embedding(
            heterogeneity
        )

        # Combine global conditions
        condition_emb = t_emb + h_emb

        # -------------------------
        # 4. UNet
        # -------------------------

        x = self.input_conv(x)

        skip1, x = self.down1(x, condition_emb)
        skip2, x = self.down2(x, condition_emb)
        skip3, x = self.down3(x, condition_emb)

        x = self.mid(x, condition_emb)

        x = self.up3(x, skip3, condition_emb)
        x = self.up2(x, skip2, condition_emb)
        x = self.up1(x, skip1, condition_emb)

        x = self.output_conv(x)

        return x

In [18]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = ConditionalUNet3D(
    image_channels=1,
    mask_channels=3,
    out_channels=1,
    base_channels=16,
    time_dim=128
).to(device)

print(
    "Number of parameters:",
    sum(p.numel() for p in model.parameters())
)

Number of parameters: 4542385


In [19]:
sample = train_dataset[0]

image = sample["image"].unsqueeze(0).to(device)
mask = sample["mask"].unsqueeze(0).to(device)
heterogeneity = sample["heterogeneity"].unsqueeze(0).to(device)

# Random diffusion timestep
t = torch.randint(
    0,
    timesteps,
    (1,),
    device=device
)

# For this smoke test, use the image as x input
with torch.no_grad():
    output = model(
        image,
        t,
        mask,
        heterogeneity
    )

print("Image shape:", image.shape)
print("Mask shape:", mask.shape)
print("Heterogeneity shape:", heterogeneity.shape)
print("Output shape:", output.shape)

Image shape: torch.Size([1, 1, 208, 224, 160])
Mask shape: torch.Size([1, 1, 208, 224, 160])
Heterogeneity shape: torch.Size([1])
Output shape: torch.Size([1, 1, 208, 224, 160])


In [51]:
def q_sample(x0, t, noise=None):
    if noise is None:
        noise = torch.randn_like(x0)

    device = x0.device

    sqrt_alpha_cumprod_device = sqrt_alphas_cumprod.to(device)
    sqrt_one_minus_alpha_cumprod_device = (
        sqrt_one_minus_alphas_cumprod.to(device)
    )

    sqrt_alpha_hat = (
        sqrt_alpha_cumprod_device[t]
        .view(-1, 1, 1, 1, 1)
    )

    sqrt_one_minus_alpha_hat = (
        sqrt_one_minus_alpha_cumprod_device[t]
        .view(-1, 1, 1, 1, 1)
    )

    xt = (
        sqrt_alpha_hat * x0
        + sqrt_one_minus_alpha_hat * noise
    )

    return xt, noise

In [16]:
def save_checkpoint(model, optimizer, epoch, path):
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict()
    }, path)


def load_checkpoint(model, optimizer, path, device):
    checkpoint = torch.load(path, map_location=device)

    model.load_state_dict(checkpoint["model_state_dict"])

    if optimizer is not None:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    return checkpoint["epoch"]

In [22]:
def train_ddpm(
    model,
    train_loader,
    epochs,
    optimizer,
    device,
    checkpoint_dir="conditional_checkpoints"
):
    import os

    os.makedirs(checkpoint_dir, exist_ok=True)

    model.train()

    loss_history = []

    # Move diffusion schedule to the same device once
    sqrt_alpha_cumprod_device = sqrt_alphas_cumprod.to(device)
    sqrt_one_minus_alpha_cumprod_device = (
        sqrt_one_minus_alphas_cumprod.to(device)
    )

    # Training-set Shannon entropy statistics
    entropy_mean = 6.870206
    entropy_std = 0.3320367

    for epoch in range(epochs):
        epoch_loss = 0.0

        for batch_idx, batch in enumerate(train_loader):

            # T2f image
            x0 = batch["image"].to(device)

            # Conditional inputs
            mask = batch["mask"].to(device)
            heterogeneity = batch["heterogeneity"].to(device)

            # Z-score normalization of Shannon entropy
            heterogeneity = (
                heterogeneity - entropy_mean
            ) / entropy_std

            # Random diffusion timestep
            t = torch.randint(
                0,
                timesteps,
                (x0.shape[0],),
                device=device
            )

            # Random Gaussian noise
            noise = torch.randn_like(x0)

            sqrt_alpha_hat = (
                sqrt_alpha_cumprod_device[t]
                .view(-1, 1, 1, 1, 1)
            )

            sqrt_one_minus_alpha_hat = (
                sqrt_one_minus_alpha_cumprod_device[t]
                .view(-1, 1, 1, 1, 1)
            )

            # Forward diffusion
            xt = (
                sqrt_alpha_hat * x0
                + sqrt_one_minus_alpha_hat * noise
            )

            # Conditional noise prediction
            predicted_noise = model(
                xt,
                t,
                mask,
                heterogeneity
            )

            # DDPM noise-prediction loss
            loss = F.mse_loss(
                predicted_noise,
                noise
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

            if (batch_idx + 1) % 10 == 0:
                print(
                    f"Epoch {epoch + 1}/{epochs} | "
                    f"Batch {batch_idx + 1}/{len(train_loader)} | "
                    f"Loss: {loss.item():.4f}"
                )

        avg_loss = epoch_loss / len(train_loader)

        loss_history.append(avg_loss)

        print(
            f"Epoch {epoch + 1} completed | "
            f"Average loss: {avg_loss:.4f}"
        )

        # Save checkpoint after every epoch
        checkpoint_path = os.path.join(
            checkpoint_dir,
            f"conditional_ddpm_epoch_{epoch + 1:03d}.pt"
        )

        save_checkpoint(
            model=model,
            optimizer=optimizer,
            epoch=epoch + 1,
            path=checkpoint_path
        )

        print("Saved:", checkpoint_path)

        np.save(
            os.path.join(
                checkpoint_dir,
                "conditional_ddpm_loss_history.npy"
            ),
            np.array(loss_history)
        )

In [49]:
@torch.no_grad()
def sample_conditional_ddpm(
    model,
    shape,
    mask,
    heterogeneity,
    device
):
    model.eval()

    # Move conditions to device
    mask = mask.to(device)

    heterogeneity = (
        heterogeneity
        .to(device)
        .float()
    )

    # Same entropy normalization as training
    entropy_mean = 6.870206
    entropy_std = 0.3320367

    heterogeneity = (
        heterogeneity - entropy_mean
    ) / entropy_std

    # Move diffusion schedule to device once
    betas_device = betas.to(device)
    alphas_device = alphas.to(device)
    alphas_cumprod_device = (
        alphas_cumprod.to(device)
    )

    # Start from Gaussian noise
    x = torch.randn(
        shape,
        device=device
    )

    for t in reversed(range(timesteps)):

        t_batch = torch.full(
            (shape[0],),
            t,
            device=device,
            dtype=torch.long
        )

        beta_t = betas_device[t]
        alpha_t = alphas_device[t]
        alpha_hat_t = (
            alphas_cumprod_device[t]
        )

        # Predict Gaussian noise
        predicted_noise = model(
            x,
            t_batch,
            mask,
            heterogeneity
        )

        # ---------------------------------
        # Predict clean image x0
        # ---------------------------------

        x0_pred = (
            x
            - torch.sqrt(
                1.0 - alpha_hat_t
            ) * predicted_noise
        ) / torch.sqrt(
            alpha_hat_t
        )

        # Training images are in [-1, 1]
        x0_pred = torch.clamp(
            x0_pred,
            -1.0,
            1.0
        )

        # ---------------------------------
        # Posterior mean
        # q(x_{t-1} | x_t, x0_pred)
        # ---------------------------------

        if t > 0:
            alpha_hat_prev = (
                alphas_cumprod_device[t - 1]
            )
        else:
            alpha_hat_prev = torch.tensor(
                1.0,
                device=device
            )

        coef_x0 = (
            torch.sqrt(alpha_hat_prev)
            * beta_t
            / (1.0 - alpha_hat_t)
        )

        coef_xt = (
            torch.sqrt(alpha_t)
            * (1.0 - alpha_hat_prev)
            / (1.0 - alpha_hat_t)
        )

        model_mean = (
            coef_x0 * x0_pred
            + coef_xt * x
        )

        # ---------------------------------
        # Sample previous timestep
        # ---------------------------------

        if t > 0:

            posterior_variance = (
                beta_t
                * (1.0 - alpha_hat_prev)
                / (1.0 - alpha_hat_t)
            )

            noise = torch.randn_like(x)

            x = (
                model_mean
                + torch.sqrt(
                    posterior_variance
                ) * noise
            )

        else:
            x = model_mean

    return x


In [42]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = ConditionalUNet3D(
    image_channels=1,
    mask_channels=3,
    out_channels=1,
    base_channels=16,
    time_dim=128
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

In [ ]:
train_ddpm(
    model=model,
    train_loader=train_loader,
    epochs=10,
    optimizer=optimizer,
    device=device,
    checkpoint_dir="conditional_v2_checkpoints"
)